In [7]:
"""
Proper Train/Validation/Test Split for Brain Tumor MRI Dataset
================================================================
This script performs a clean 70/15/15 split of your dataset.

Input: Preprocessed/Augmented images organized by class
Output: Three separate folders (train/val/test) with proper stratified split

Author: Your Name
Date: 2024
"""

import os
import shutil
import random
from pathlib import Path
from collections import defaultdict
import json
from datetime import datetime


def set_random_seed(seed=42):
    """Set random seed for reproducibility"""
    random.seed(seed)


def validate_input_directory(input_dir):
    """
    Validate that input directory exists and has the expected structure
    
    Args:
        input_dir (Path): Input directory path
        
    Returns:
        bool: True if valid, False otherwise
    """
    if not input_dir.exists():
        print(f"❌ Error: Input directory '{input_dir}' does not exist!")
        return False
    
    if not input_dir.is_dir():
        print(f"❌ Error: '{input_dir}' is not a directory!")
        return False
    
    return True


def get_class_folders(input_dir):
    """
    Get all class folders from input directory
    
    Args:
        input_dir (Path): Input directory path
        
    Returns:
        list: List of class folder names
    """
    # Expected 10 classes
    expected_classes = [
        'Astrocytoma',
        'Ependymoma',
        'Glioblastoma',
        'Germinoma',
        'Medulloblastoma',
        'Meningioma',
        'Oligodendroglioma',
        'Pituitary',
        'Schwannoma',
        'No_Tumor'
    ]
    
    # Find which classes actually exist
    found_classes = []
    for class_name in expected_classes:
        class_path = input_dir / class_name
        if class_path.exists() and class_path.is_dir():
            found_classes.append(class_name)
        else:
            print(f"⚠️  Warning: Class '{class_name}' not found in input directory")
    
    if not found_classes:
        print("❌ Error: No valid class folders found!")
        return None
    
    print(f"✅ Found {len(found_classes)} classes: {', '.join(found_classes)}")
    return found_classes


def get_image_files(class_dir):
    """
    Get all image files from a class directory
    
    Args:
        class_dir (Path): Path to class directory
        
    Returns:
        list: List of image file paths
    """
    image_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}
    image_files = []
    
    for file in class_dir.iterdir():
        if file.is_file() and file.suffix.lower() in image_extensions:
            image_files.append(file)
    
    return image_files


def split_files(files, train_ratio=0.70, val_ratio=0.15, test_ratio=0.15):
    """
    Split files into train/val/test sets
    
    Args:
        files (list): List of file paths
        train_ratio (float): Training set ratio
        val_ratio (float): Validation set ratio
        test_ratio (float): Test set ratio
        
    Returns:
        tuple: (train_files, val_files, test_files)
    """
    # Shuffle files
    files = list(files)
    random.shuffle(files)
    
    # Calculate split indices
    total = len(files)
    train_end = int(total * train_ratio)
    val_end = train_end + int(total * val_ratio)
    
    # Split
    train_files = files[:train_end]
    val_files = files[train_end:val_end]
    test_files = files[val_end:]
    
    return train_files, val_files, test_files


def create_output_structure(output_dir, classes):
    """
    Create output directory structure for train/val/test
    
    Args:
        output_dir (Path): Output directory path
        classes (list): List of class names
    """
    for split in ['train', 'val', 'test']:
        for class_name in classes:
            split_class_dir = output_dir / split / class_name
            split_class_dir.mkdir(parents=True, exist_ok=True)


def copy_files(file_list, dest_dir):
    """
    Copy files to destination directory
    
    Args:
        file_list (list): List of source file paths
        dest_dir (Path): Destination directory
        
    Returns:
        int: Number of files copied
    """
    count = 0
    for file_path in file_list:
        dest_path = dest_dir / file_path.name
        shutil.copy2(file_path, dest_path)
        count += 1
    return count


def clean_output_directory(output_dir):
    """
    Remove existing output directory
    
    Args:
        output_dir (Path): Output directory to clean
    """
    if output_dir.exists():
        print(f"🗑️  Removing existing output directory...")
        shutil.rmtree(output_dir)


def save_split_summary(output_dir, stats, ratios, seed):
    """
    Save split summary to JSON file
    
    Args:
        output_dir (Path): Output directory
        stats (dict): Split statistics
        ratios (dict): Split ratios used
        seed (int): Random seed used
    """
    summary = {
        'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'random_seed': seed,
        'split_ratios': ratios,
        'statistics': stats
    }
    
    summary_file = output_dir / 'split_summary.json'
    with open(summary_file, 'w') as f:
        json.dump(summary, f, indent=2)
    
    print(f"\n💾 Summary saved to: {summary_file}")


def perform_train_val_test_split(input_dir, output_dir, train_ratio=0.70, val_ratio=0.15, 
                                   test_ratio=0.15, random_seed=42):
    """
    Main function to perform train/val/test split
    
    Args:
        input_dir (str): Path to input directory with class folders
        output_dir (str): Path to output directory
        train_ratio (float): Training set ratio (default: 0.70)
        val_ratio (float): Validation set ratio (default: 0.15)
        test_ratio (float): Test set ratio (default: 0.15)
        random_seed (int): Random seed for reproducibility (default: 42)
    """
    # Convert to Path objects
    input_dir = Path(input_dir)
    output_dir = Path(output_dir)
    
    # Validate ratios
    total_ratio = train_ratio + val_ratio + test_ratio
    if abs(total_ratio - 1.0) > 0.001:
        print(f"❌ Error: Ratios must sum to 1.0, got {total_ratio}")
        return False
    
    # Print header
    print("\n" + "=" * 80)
    print("🔀 TRAIN/VAL/TEST SPLIT")
    print("=" * 80)
    print(f"Input:  {input_dir}")
    print(f"Output: {output_dir}")
    print(f"Split:  Train={train_ratio*100:.0f}% | Val={val_ratio*100:.0f}% | Test={test_ratio*100:.0f}%")
    print(f"Seed:   {random_seed}")
    print("=" * 80 + "\n")
    
    # Set random seed
    set_random_seed(random_seed)
    
    # Validate input directory
    if not validate_input_directory(input_dir):
        return False
    
    # Get class folders
    classes = get_class_folders(input_dir)
    if not classes:
        return False
    
    # Clean and create output structure
    clean_output_directory(output_dir)
    print("📁 Creating output directory structure...")
    create_output_structure(output_dir, classes)
    
    # Statistics tracking
    stats = defaultdict(lambda: defaultdict(int))
    
    # Process each class
    print("\n" + "=" * 80)
    print("📊 PROCESSING CLASSES")
    print("=" * 80 + "\n")
    
    for class_name in classes:
        print(f"Class: {class_name}")
        
        # Get all images for this class
        class_dir = input_dir / class_name
        image_files = get_image_files(class_dir)
        
        if not image_files:
            print(f"  ⚠️  No images found, skipping...\n")
            continue
        
        print(f"  Total images: {len(image_files)}")
        
        # Split files
        train_files, val_files, test_files = split_files(
            image_files, train_ratio, val_ratio, test_ratio
        )
        
        # Copy files to respective directories
        train_dest = output_dir / 'train' / class_name
        val_dest = output_dir / 'val' / class_name
        test_dest = output_dir / 'test' / class_name
        
        train_count = copy_files(train_files, train_dest)
        val_count = copy_files(val_files, val_dest)
        test_count = copy_files(test_files, test_dest)
        
        # Update statistics
        stats[class_name]['train'] = train_count
        stats[class_name]['val'] = val_count
        stats[class_name]['test'] = test_count
        stats[class_name]['total'] = len(image_files)
        
        print(f"  ✓ Train: {train_count} | Val: {val_count} | Test: {test_count}\n")
    
    # Print final statistics
    print("=" * 80)
    print("📈 FINAL STATISTICS")
    print("=" * 80 + "\n")
    
    print(f"{'Class':<20} {'Train':<10} {'Val':<10} {'Test':<10} {'Total':<10}")
    print("-" * 70)
    
    total_train = 0
    total_val = 0
    total_test = 0
    total_all = 0
    
    for class_name in sorted(classes):
        train = stats[class_name]['train']
        val = stats[class_name]['val']
        test = stats[class_name]['test']
        total = stats[class_name]['total']
        
        print(f"{class_name:<20} {train:<10} {val:<10} {test:<10} {total:<10}")
        
        total_train += train
        total_val += val
        total_test += test
        total_all += total
    
    print("-" * 70)
    print(f"{'TOTAL':<20} {total_train:<10} {total_val:<10} {total_test:<10} {total_all:<10}")
    
    # Print actual ratios
    print("\n📊 Actual Split Percentages:")
    if total_all > 0:
        print(f"  Train: {total_train/total_all*100:.2f}%")
        print(f"  Val:   {total_val/total_all*100:.2f}%")
        print(f"  Test:  {total_test/total_all*100:.2f}%")
    
    # Verification
    print("\n✅ Verification:")
    output_total = total_train + total_val + total_test
    print(f"  Input total:  {total_all}")
    print(f"  Output total: {output_total}")
    
    if total_all == output_total:
        print("  ✓ Perfect! No images lost or duplicated!")
    else:
        print(f"  ⚠️  Warning: Mismatch of {abs(total_all - output_total)} images")
    
    print("=" * 80)
    
    # Save summary
    ratios = {
        'train': train_ratio,
        'val': val_ratio,
        'test': test_ratio
    }
    save_split_summary(output_dir, dict(stats), ratios, random_seed)
    
    # Success message
    print("\n✅ Split completed successfully!")
    print(f"📂 Output saved to: {output_dir}")
    print("\n⚠️  IMPORTANT:")
    print("  - Use 'train' folder for model training")
    print("  - Use 'val' folder for hyperparameter tuning")
    print("  - Use 'test' folder ONLY for final evaluation")
    print("\n" + "=" * 80 + "\n")
    
    return True


# =============================================================================
# MAIN EXECUTION
# =============================================================================

if __name__ == "__main__":
    
    # -------------------------------------------------------------------------
    # CONFIGURATION - CHANGE THESE PATHS
    # -------------------------------------------------------------------------
    
    INPUT_DIRECTORY = "E:/Semisters/brain tumer prediction/brain_mri_10class_augmented"
    OUTPUT_DIRECTORY = "data"
    
    # Split ratios (must sum to 1.0)
    TRAIN_RATIO = 0.70    # 70% for training
    VAL_RATIO = 0.15      # 15% for validation
    TEST_RATIO = 0.15     # 15% for testing
    
    # Random seed for reproducibility
    RANDOM_SEED = 42
    
    # -------------------------------------------------------------------------
    # RUN THE SPLIT
    # -------------------------------------------------------------------------
    
    success = perform_train_val_test_split(
        input_dir=INPUT_DIRECTORY,
        output_dir=OUTPUT_DIRECTORY,
        train_ratio=TRAIN_RATIO,
        val_ratio=VAL_RATIO,
        test_ratio=TEST_RATIO,
        random_seed=RANDOM_SEED
    )
    
    if success:
        print("🎉 All done! Ready to train your model.")
    else:
        print("❌ Split failed. Please check the errors above.")


🔀 TRAIN/VAL/TEST SPLIT
Input:  E:\Semisters\brain tumer prediction\brain_mri_10class_augmented
Output: data
Split:  Train=70% | Val=15% | Test=15%
Seed:   42

✅ Found 10 classes: Astrocytoma, Ependymoma, Glioblastoma, Germinoma, Medulloblastoma, Meningioma, Oligodendroglioma, Pituitary, Schwannoma, No_Tumor
📁 Creating output directory structure...

📊 PROCESSING CLASSES

Class: Astrocytoma
  Total images: 1287
  ✓ Train: 900 | Val: 193 | Test: 194

Class: Ependymoma
  Total images: 1850
  ✓ Train: 1295 | Val: 277 | Test: 278

Class: Glioblastoma
  Total images: 1796
  ✓ Train: 1257 | Val: 269 | Test: 270

Class: Germinoma
  Total images: 1900
  ✓ Train: 1330 | Val: 285 | Test: 285

Class: Medulloblastoma
  Total images: 1869
  ✓ Train: 1308 | Val: 280 | Test: 281

Class: Meningioma
  Total images: 2582
  ✓ Train: 1807 | Val: 387 | Test: 388

Class: Oligodendroglioma
  Total images: 1776
  ✓ Train: 1243 | Val: 266 | Test: 267

Class: Pituitary
  Total images: 2658
  ✓ Train: 1860 | Val: